In [1]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path

airkorea_dir = Path("/Users/drewbaldwin/PM2_5 Research/airkorea")

csv_paths = sorted(airkorea_dir.glob("*/*.csv"))
print(f"Found {len(csv_paths)} sensor CSVs across "
      f"{len({p.parent.name for p in csv_paths})} years")

Found 2430 sensor CSVs across 6 years


In [4]:
POLLUTANT_DTYPES = {
    "SO2": "float32",
    "CO": "float32",
    "O3": "float32",
    "NO2": "float32",
    "PM10": "float32",
    "PM25": "float32",
}

def read_airkorea_csv(path: Path) -> pd.DataFrame:
    # Station_ID is dropped from the CSV and taken from the filename instead: some
    # station/year files are entirely empty (no readings that year), which leaves
    # Station_ID all-NaN and unable to hold an int dtype.
    df = pd.read_csv(
        path,
        index_col=0,
        parse_dates=True,
        dtype=POLLUTANT_DTYPES,
        usecols=lambda c: c != "Station_ID",
    )
    df.index.name = "Datetime"
    df["Station_ID"] = int(path.stem)
    df["Year"] = path.parent.name
    return df

dfs = [read_airkorea_csv(p) for p in csv_paths]
air_korea = pd.concat(dfs, ignore_index=False)
air_korea = air_korea.reset_index()
del dfs

air_korea.shape

(21306240, 9)

In [5]:
air_korea = air_korea.sort_values(["Station_ID", "Datetime"]).reset_index(drop=True)
air_korea.info()
air_korea.head()

<class 'pandas.DataFrame'>
RangeIndex: 21306240 entries, 0 to 21306239
Data columns (total 9 columns):
 #   Column      Dtype         
---  ------      -----         
 0   Datetime    datetime64[us]
 1   SO2         float32       
 2   CO          float32       
 3   O3          float32       
 4   NO2         float32       
 5   PM10        float32       
 6   PM25        float32       
 7   Station_ID  int64         
 8   Year        str           
dtypes: datetime64[us](1), float32(6), int64(1), str(1)
memory usage: 1.0 GB


,Datetime,SO2,CO,O3,NO2,PM10,PM25,Station_ID,Year
0,2016-01-01 00:00:00,7.0,1000.0,2.0,76.0,77.0,53.0,111121,2016
1,2016-01-01 01:00:00,7.0,1100.0,2.0,77.0,70.0,48.0,111121,2016
2,2016-01-01 02:00:00,7.0,1200.0,2.0,78.0,75.0,53.0,111121,2016
3,2016-01-01 03:00:00,6.0,1400.0,2.0,78.0,77.0,53.0,111121,2016
4,2016-01-01 04:00:00,6.0,1500.0,2.0,77.0,83.0,52.0,111121,2016


In [6]:
air_korea.to_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_combined.pkl")

In [7]:
pm25_missing_pct = (
    air_korea.groupby("Station_ID")["PM25"]
    .apply(lambda s: s.isna().mean())
    .sort_values(ascending=False)
    .rename("pct_missing")
    .to_frame()
)
pm25_missing_pct["pct_missing"] = (pm25_missing_pct["pct_missing"] * 100).round(2)

print(f"{len(pm25_missing_pct)} stations total")
pm25_missing_pct.describe()


405 stations total


,pct_missing
count,405.000000
mean,27.905531
std,21.095464
min,1.010000
25%,7.190000
50%,25.880000
75%,44.840000
max,83.100000


In [8]:
MAX_PM25_MISSING_PCT = 20.0

keep_stations = pm25_missing_pct[pm25_missing_pct["pct_missing"] <= MAX_PM25_MISSING_PCT].index
dropped_stations = pm25_missing_pct[pm25_missing_pct["pct_missing"] > MAX_PM25_MISSING_PCT].index

print(f"Keeping {len(keep_stations)} stations, dropping {len(dropped_stations)} "
      f"stations with > {MAX_PM25_MISSING_PCT}% missing PM2.5")

air_korea_filtered = (
    air_korea[air_korea["Station_ID"].isin(keep_stations)]
    .sort_values(["Station_ID", "Datetime"])
    .reset_index(drop=True)
)

print(f"{len(air_korea_filtered):,} of {len(air_korea):,} rows retained")
air_korea_filtered.isna().mean().sort_values(ascending=False) * 100


Keeping 176 stations, dropping 229 stations with > 20.0% missing PM2.5
9,259,008 of 21,306,240 rows retained


PM25          7.208591
CO            5.356211
SO2           5.174917
PM10          4.839806
NO2           3.771786
O3            3.435098
Datetime      0.000000
Station_ID    0.000000
Year          0.000000
dtype: float64

In [9]:
air_korea_filtered.to_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_pm25_filtered.pkl")


In [10]:
#merging with list of stations from air korea to get locations.

stations = pd.read_csv("/Users/drewbaldwin/PM2_5 Research/airkorea_stations_2019.csv")
stations = stations[["station_code", "lon", "lat"]].rename(columns={"station_code": "Station_ID"})

assert not stations["Station_ID"].duplicated().any(), "duplicate station_code in metadata file"
assert stations[["lon", "lat"]].notna().all().all(), "missing lon/lat in metadata file"

stations.head()

sensor_ids = set(air_korea_filtered["Station_ID"])
station_ids = set(stations["Station_ID"])

missing_metadata = sensor_ids - station_ids
extra_metadata = station_ids - sensor_ids

print(f"{len(sensor_ids)} stations in sensor data, {len(station_ids)} in metadata")
print(f"sensor stations missing from metadata: {len(missing_metadata)}")
print(f"metadata stations not present in sensor data (will be dropped by the merge): {len(extra_metadata)}")

assert not missing_metadata, f"these Station_IDs have no lat/lon match: {sorted(missing_metadata)}"

rows_before = len(air_korea_filtered)

air_korea_geo = air_korea_filtered.merge(stations, on="Station_ID", how="left", validate="many_to_one")

assert len(air_korea_geo) == rows_before, "merge changed row count"
assert air_korea_geo[["lon", "lat"]].notna().all().all(), "merge produced NaN lon/lat"

print(f"{len(air_korea_geo):,} rows, {air_korea_geo['Station_ID'].nunique()} stations, "
      f"columns: {air_korea_geo.columns.tolist()}")
air_korea_geo.head()



176 stations in sensor data, 451 in metadata
sensor stations missing from metadata: 0
metadata stations not present in sensor data (will be dropped by the merge): 275
9,259,008 rows, 176 stations, columns: ['Datetime', 'SO2', 'CO', 'O3', 'NO2', 'PM10', 'PM25', 'Station_ID', 'Year', 'lon', 'lat']


,Datetime,SO2,CO,O3,NO2,PM10,PM25,Station_ID,Year,lon,lat
0,2016-01-01 00:00:00,7.0,1000.0,2.0,76.0,77.0,53.0,111121,2016,126.9747,37.5643
1,2016-01-01 01:00:00,7.0,1100.0,2.0,77.0,70.0,48.0,111121,2016,126.9747,37.5643
2,2016-01-01 02:00:00,7.0,1200.0,2.0,78.0,75.0,53.0,111121,2016,126.9747,37.5643
3,2016-01-01 03:00:00,6.0,1400.0,2.0,78.0,77.0,53.0,111121,2016,126.9747,37.5643
4,2016-01-01 04:00:00,6.0,1500.0,2.0,77.0,83.0,52.0,111121,2016,126.9747,37.5643


In [11]:
air_korea_geo.to_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_pm25_filtered_geo.pkl")


In [ ]:
from pathlib import Path
import time
import requests
import pandas as pd
from tqdm.auto import tqdm

# Load the PM2.5 + station geo dataset (built earlier, before the Open-Meteo step)
air_korea_geo = pd.read_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_pm25_filtered_geo.pkl")

# Reconstruct the station metadata + kept-station list from it
stations = pd.read_csv("/Users/drewbaldwin/PM2_5 Research/airkorea_stations_2019.csv")
stations = stations[["station_code", "lon", "lat"]].rename(columns={"station_code": "Station_ID"})

keep_stations = air_korea_geo["Station_ID"].unique()

# Open-Meteo pull config (same as before)
OPENMETEO_URL = "https://archive-api.open-meteo.com/v1/archive"
OPENMETEO_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "windspeed_10m",
    "winddirection_10m",
    "precipitation",
]
YEARS = list(range(2016, 2022))

openmeteo_cache_dir = Path("/Users/drewbaldwin/PM2_5 Research/openmeteo_cache")
openmeteo_cache_dir.mkdir(exist_ok=True)

pull_stations = (
    stations[stations["Station_ID"].isin(keep_stations)]
    .drop_duplicates("Station_ID")
    .reset_index(drop=True)
)
print(f"{len(pull_stations)} stations, {len(pull_stations) * len(YEARS)} station-years to pull")

cached = len(list(openmeteo_cache_dir.glob("*.csv")))
print(f"{cached} station-years already cached, {len(pull_stations) * len(YEARS) - cached} remaining")


## Pull Open-Meteo weather predictors

The `wrf`/`cmaq` folders (from the TransNet/AGATNet Zenodo dataset, 10.5281/zenodo.10963116) turned
out to be keyed by WRF grid index, not station ID or lat/lon, and no mapping file was ever
published for it — so we can't join those predictors to our stations. Instead we pull weather
directly from the Open-Meteo historical archive API at each station's exact lat/lon (no
grid-approximation needed), for the same 2016-2021 hourly range as the PM2.5 data.

Open-Meteo's free tier is rate-limited (per-minute request quota), so this fetches one
station-year at a time, caches each response to disk, and retries with backoff on HTTP 429 —
safe to re-run if interrupted, since cached station-years are skipped.

In [12]:
import time
import requests
from tqdm.auto import tqdm

OPENMETEO_URL = "https://archive-api.open-meteo.com/v1/archive"
OPENMETEO_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "windspeed_10m",
    "winddirection_10m",
    "precipitation",
]
YEARS = list(range(2016, 2022))

openmeteo_cache_dir = Path("/Users/drewbaldwin/PM2_5 Research/openmeteo_cache")
openmeteo_cache_dir.mkdir(exist_ok=True)

pull_stations = (
    stations[stations["Station_ID"].isin(keep_stations)]
    .drop_duplicates("Station_ID")
    .reset_index(drop=True)
)
print(f"{len(pull_stations)} stations, {len(pull_stations) * len(YEARS)} station-years to pull")

176 stations, 1056 station-years to pull


In [ ]:
def fetch_station_year(station_id: int, lat: float, lon: float, year: int) -> pd.DataFrame:
    cache_path = openmeteo_cache_dir / f"{station_id}_{year}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path, parse_dates=["Datetime"])

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": f"{year}-01-01",
        "end_date": f"{year}-12-31",
        "hourly": ",".join(OPENMETEO_VARS),
        "timezone": "Asia/Seoul",
    }

    for attempt in range(6):
        resp = requests.get(OPENMETEO_URL, params=params, timeout=30)
        if resp.status_code == 429:
            reason = resp.json().get("reason", "")
            if "daily" in reason.lower():
                raise RuntimeError(
                    f"Open-Meteo daily quota exhausted (station {station_id} {year}): {reason}. "
                    "Progress so far is cached -- wait for the quota to reset (next UTC day) "
                    "and re-run this cell to resume."
                )
            wait = 60 * (attempt + 1)
            print(f"rate limited on station {station_id} {year}, waiting {wait}s")
            time.sleep(wait)
            continue
        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f"failed to fetch station {station_id} year {year} after retries")

    df = pd.DataFrame(resp.json()["hourly"]).rename(columns={"time": "Datetime"})
    df["Datetime"] = pd.to_datetime(df["Datetime"])
    df["Station_ID"] = station_id
    df.to_csv(cache_path, index=False)
    time.sleep(1.5)  # stay under Open-Meteo's per-minute quota
    return df

In [14]:
results = []
for _, row in tqdm(pull_stations.iterrows(), total=len(pull_stations), desc="stations"):
    for year in YEARS:
        results.append(fetch_station_year(int(row["Station_ID"]), row["lat"], row["lon"], year))

openmeteo_data = pd.concat(results, ignore_index=True)
del results

assert not openmeteo_data.duplicated(["Station_ID", "Datetime"]).any(), "duplicate station/datetime rows"
print(openmeteo_data.shape)
openmeteo_data.head()

stations:  23%|██▎       | 41/176 [11:12<36:53, 16.40s/it]

rate limited on station 221233 2019, waiting 60s
rate limited on station 221233 2019, waiting 120s
rate limited on station 221233 2019, waiting 180s
rate limited on station 221233 2019, waiting 240s
rate limited on station 221233 2019, waiting 300s


stations:  53%|█████▎    | 94/176 [41:12<22:24, 16.39s/it]    

rate limited on station 131132 2020, waiting 60s
rate limited on station 131132 2020, waiting 120s
rate limited on station 131132 2020, waiting 180s
rate limited on station 131132 2020, waiting 240s
rate limited on station 131132 2020, waiting 300s
rate limited on station 131132 2020, waiting 360s


stations:  53%|█████▎    | 94/176 [1:02:27<54:29, 39.87s/it]


RuntimeError: failed to fetch station 131132 year 2020 after retries

In [ ]:
rows_before = len(air_korea_geo)

air_korea_full = air_korea_geo.merge(
    openmeteo_data,
    on=["Station_ID", "Datetime"],
    how="left",
    validate="one_to_one",
)

assert len(air_korea_full) == rows_before, "merge changed row count"

missing_weather_pct = air_korea_full[OPENMETEO_VARS].isna().any(axis=1).mean() * 100
print(f"{len(air_korea_full):,} rows, {missing_weather_pct:.2f}% missing at least one weather var")
air_korea_full.head()

In [ ]:
air_korea_full.to_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_full.pkl")

## Add land use / elevation predictors

These are static (per-station, time-invariant) features, so they don't depend on the Open-Meteo
pull finishing. Reusing the same recipe already proven in `korea_application_data_prep.ipynb`:
elevation from the Open-Meteo Elevation API, and land-use/road/building/railway features from the
local OSM extract `south-korea.gpkg`, computed within a 3km buffer around each station.

Merged onto `air_korea_geo` (PM2.5 + station lat/lon) since `air_korea_full` isn't built yet —
once the weather pull finishes, merge `sensor_static` onto `air_korea_full` the same way.

In [ ]:
import geopandas as gpd

GPKG_PATH = "/Users/drewbaldwin/PM2_5 Research/south-korea.gpkg"
BUFFER_M = 3000

stations_unique = air_korea_geo[["Station_ID", "lon", "lat"]].drop_duplicates().reset_index(drop=True)
print(f"{len(stations_unique)} stations")

# --- Elevation (Open-Meteo Elevation API, batched up to 100 points) ---
elevations = []
for i in range(0, len(stations_unique), 100):
    chunk = stations_unique.iloc[i : i + 100]
    lats = ",".join(chunk["lat"].astype(str))
    lons = ",".join(chunk["lon"].astype(str))
    resp = requests.get(f"https://api.open-meteo.com/v1/elevation?latitude={lats}&longitude={lons}")
    resp.raise_for_status()
    elevations.extend(resp.json()["elevation"])
stations_unique["elevation_m"] = elevations
print(stations_unique["elevation_m"].describe())

In [ ]:
# --- Buffer each station 3km, project to a metric CRS for South Korea (EPSG:5179) ---
sensors_gdf = gpd.GeoDataFrame(
    stations_unique,
    geometry=gpd.points_from_xy(stations_unique["lon"], stations_unique["lat"]),
    crs="EPSG:4326",
).to_crs(epsg=5179)
sensors_gdf["buffer_geom"] = sensors_gdf.geometry.buffer(BUFFER_M)
sensors_buffered = sensors_gdf.set_geometry("buffer_geom")[["Station_ID", "buffer_geom"]]

# --- Load OSM layers, cropped to the stations' bounding box for speed ---
lon_pad, lat_pad = 0.05, 0.05
bbox = (
    stations_unique["lon"].min() - lon_pad, stations_unique["lat"].min() - lat_pad,
    stations_unique["lon"].max() + lon_pad, stations_unique["lat"].max() + lat_pad,
)

roads_gdf = gpd.read_file(GPKG_PATH, layer="gis_osm_roads_free", bbox=bbox).to_crs(epsg=5179)
landuse_gdf = gpd.read_file(GPKG_PATH, layer="gis_osm_landuse_a_free", bbox=bbox).to_crs(epsg=5179)
buildings_gdf = gpd.read_file(GPKG_PATH, layer="gis_osm_buildings_a_free", bbox=bbox).to_crs(epsg=5179)
railways_gdf = gpd.read_file(GPKG_PATH, layer="gis_osm_railways_free", bbox=bbox).to_crs(epsg=5179)
print(f"{len(roads_gdf):,} roads, {len(landuse_gdf):,} landuse polys, "
      f"{len(buildings_gdf):,} buildings, {len(railways_gdf):,} railways")

In [ ]:
# --- Roads: count of major roads intersecting the buffer, total road length within it ---
major_roads = roads_gdf[roads_gdf["fclass"].isin(["motorway", "trunk", "primary"])]
major_join = gpd.sjoin(major_roads, sensors_buffered, predicate="intersects", how="inner")
road_counts = major_join.groupby("Station_ID").size().reset_index(name="major_roads_count_3km")

road_overlay = gpd.overlay(roads_gdf, sensors_buffered, how="intersection")
road_overlay["road_length_m"] = road_overlay.geometry.length
total_road_length = road_overlay.groupby("Station_ID")["road_length_m"].sum().reset_index(name="total_road_length_3km")

# --- Land use: urban (residential/commercial/industrial/retail) and green space area ---
urban_landuse = landuse_gdf[landuse_gdf["fclass"].isin(["residential", "commercial", "industrial", "retail"])]
urban_overlay = gpd.overlay(urban_landuse, sensors_buffered, how="intersection")
urban_overlay["area_m2"] = urban_overlay.geometry.area
urban_area = urban_overlay.groupby("Station_ID")["area_m2"].sum().reset_index(name="urban_landuse_area_m2_3km")

green_landuse = landuse_gdf[landuse_gdf["fclass"].isin(["forest", "wood", "grass", "park"])]
green_overlay = gpd.overlay(green_landuse, sensors_buffered, how="intersection")
green_overlay["area_m2"] = green_overlay.geometry.area
green_area = green_overlay.groupby("Station_ID")["area_m2"].sum().reset_index(name="green_space_area_3km")

# --- Buildings: total footprint area within the buffer ---
building_overlay = gpd.overlay(buildings_gdf, sensors_buffered, how="intersection")
building_overlay["area_m2"] = building_overlay.geometry.area
building_area = building_overlay.groupby("Station_ID")["area_m2"].sum().reset_index(name="building_footprint_area_3km")

# --- Railways: total length within the buffer ---
rail_overlay = gpd.overlay(railways_gdf, sensors_buffered, how="intersection")
rail_overlay["rail_length_m"] = rail_overlay.geometry.length
rail_length = rail_overlay.groupby("Station_ID")["rail_length_m"].sum().reset_index(name="railway_length_3km")

# --- Combine: stations with zero intersecting features get 0, not NaN ---
sensor_static = stations_unique[["Station_ID", "elevation_m"]].copy()
for feat in [road_counts, total_road_length, urban_area, green_area, building_area, rail_length]:
    sensor_static = sensor_static.merge(feat, on="Station_ID", how="left")

zero_fill_cols = [
    "major_roads_count_3km", "total_road_length_3km", "urban_landuse_area_m2_3km",
    "green_space_area_3km", "building_footprint_area_3km", "railway_length_3km",
]
sensor_static[zero_fill_cols] = sensor_static[zero_fill_cols].fillna(0)

assert sensor_static.notna().all().all(), "unexpected NaN in sensor_static"
print(sensor_static.shape)
sensor_static.describe()

In [ ]:
rows_before = len(air_korea_geo)

air_korea_static = air_korea_geo.merge(sensor_static, on="Station_ID", how="left", validate="many_to_one")

assert len(air_korea_static) == rows_before, "merge changed row count"
static_cols = [c for c in sensor_static.columns if c != "Station_ID"]
assert air_korea_static[static_cols].notna().all().all(), "merge produced NaN static features"

print(f"{len(air_korea_static):,} rows, columns: {air_korea_static.columns.tolist()}")
air_korea_static.head()

In [ ]:
sensor_static.to_pickle("/Users/drewbaldwin/PM2_5 Research/sensor_static_features.pkl")
air_korea_static.to_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_pm25_filtered_geo_static.pkl")

## Add 4 more static predictors

- **Distance to coast**: nearest `beach` polygon in `gis_osm_natural_a_free`. The water-area layer
  (`gis_osm_water_a_free`) only has inland water (lakes/reservoirs/rivers) -- no sea polygon -- so
  distance to that layer would be misleading. Beach polygons trace the actual coastline.
- **Distance to nearest major road**: point-to-line distance to the closest motorway/trunk/primary
  road, a more direct proxy than the existing count-within-3km.
- **Industrial land area within 3km**: split out from the combined "urban" bucket, since industrial
  land is a stronger PM2.5 source signal than residential/commercial.
- **Traffic point count within 3km**: `traffic_signals`, `motorway_junction`, and `mini_roundabout`
  points from `gis_osm_traffic_free`, as a congestion/intersection-density proxy (excluding other
  fclasses in that layer like `waterfall`/`dam`/`marina`, which are water infrastructure, not
  traffic).

In [ ]:
natural_a_gdf = gpd.read_file(GPKG_PATH, layer="gis_osm_natural_a_free", bbox=bbox).to_crs(epsg=5179)
traffic_gdf = gpd.read_file(GPKG_PATH, layer="gis_osm_traffic_free", bbox=bbox).to_crs(epsg=5179)

sensors_points = sensors_gdf.set_geometry("geometry")[["Station_ID", "geometry"]]

# --- Distance to coast (nearest beach polygon) ---
beaches = natural_a_gdf[natural_a_gdf["fclass"] == "beach"]
coast_nearest = gpd.sjoin_nearest(sensors_points, beaches[["geometry"]], distance_col="dist_m")
dist_to_coast = (
    coast_nearest.groupby("Station_ID")["dist_m"].min().reset_index(name="dist_to_coast_km")
)
dist_to_coast["dist_to_coast_km"] /= 1000

# --- Distance to nearest major road ---
major_roads = roads_gdf[roads_gdf["fclass"].isin(["motorway", "trunk", "primary"])]
road_nearest = gpd.sjoin_nearest(sensors_points, major_roads[["geometry"]], distance_col="dist_m")
dist_to_road = (
    road_nearest.groupby("Station_ID")["dist_m"].min().reset_index(name="dist_to_major_road_km")
)
dist_to_road["dist_to_major_road_km"] /= 1000

# --- Industrial land area within 3km ---
industrial_landuse = landuse_gdf[landuse_gdf["fclass"] == "industrial"]
industrial_overlay = gpd.overlay(industrial_landuse, sensors_buffered, how="intersection")
industrial_overlay["area_m2"] = industrial_overlay.geometry.area
industrial_area = (
    industrial_overlay.groupby("Station_ID")["area_m2"].sum().reset_index(name="industrial_area_m2_3km")
)

# --- Traffic point count within 3km ---
TRAFFIC_FCLASSES = ["traffic_signals", "motorway_junction", "mini_roundabout"]
traffic_pts = traffic_gdf[traffic_gdf["fclass"].isin(TRAFFIC_FCLASSES)]
traffic_join = gpd.sjoin(traffic_pts, sensors_buffered, predicate="intersects", how="inner")
traffic_counts = traffic_join.groupby("Station_ID").size().reset_index(name="traffic_points_count_3km")

print(dist_to_coast["dist_to_coast_km"].describe())
print(dist_to_road["dist_to_major_road_km"].describe())
print(industrial_area["industrial_area_m2_3km"].describe())
print(traffic_counts["traffic_points_count_3km"].describe())

In [ ]:
for feat in [dist_to_coast, dist_to_road, industrial_area, traffic_counts]:
    sensor_static = sensor_static.merge(feat, on="Station_ID", how="left")

# Missing distance would mean no beach/major road anywhere in the loaded bbox -- shouldn't
# happen here, so leave those as NaN if it ever does (surfaces as a loud failure, not silent 0).
# Area/count features are legitimately 0 when nothing of that type is within the buffer.
sensor_static[["industrial_area_m2_3km", "traffic_points_count_3km"]] = (
    sensor_static[["industrial_area_m2_3km", "traffic_points_count_3km"]].fillna(0)
)

assert sensor_static[["dist_to_coast_km", "dist_to_major_road_km"]].notna().all().all(), \
    "distance features have unexpected NaN -- a station has no beach/major road in the loaded bbox"
assert sensor_static.notna().all().all(), "unexpected NaN in sensor_static"

print(sensor_static.shape)
sensor_static.describe()

In [ ]:
rows_before = len(air_korea_geo)

air_korea_static = air_korea_geo.merge(sensor_static, on="Station_ID", how="left", validate="many_to_one")

assert len(air_korea_static) == rows_before, "merge changed row count"
static_cols = [c for c in sensor_static.columns if c != "Station_ID"]
assert air_korea_static[static_cols].notna().all().all(), "merge produced NaN static features"

print(f"{len(air_korea_static):,} rows, columns: {air_korea_static.columns.tolist()}")
air_korea_static.head()

In [ ]:
sensor_static.to_pickle("/Users/drewbaldwin/PM2_5 Research/sensor_static_features.pkl")
air_korea_static.to_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_pm25_filtered_geo_static.pkl")